<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《从零构建大语言模型》（Build a Large Language Model From Scratch）</a> 一书的配套代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 第 6 章：分类微调

In [ ]:
from importlib.metadata import version

pkgs = ["matplotlib",  # 绘图库
        "numpy",       # PyTorch 与 TensorFlow 依赖
        "tiktoken",    # 分词器
        "torch",       # 深度学习库
        "tensorflow",  # 用于 OpenAI 预训练权重
        "pandas"       # 数据集加载
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/01.webp" width=500px>

&nbsp;
### 6.1 微调的不同类别

- 本节无代码

- 微调语言模型最常见的方式是指令微调（instruction-finetuning）与分类微调（classification finetuning）
- 下图所示的指令微调将在下一章讨论

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/02.webp" width=500px>

- 分类微调是本章主题；若你有机器学习背景，可能已熟悉这一流程——例如训练卷积网络识别手写数字
- 在分类微调中，模型输出固定数量的类别标签（例如「spam」与「not spam」）
- 经分类微调的模型只能预测训练时见过的类别（例如「spam」或「not spam」），而经指令微调的模型通常可执行多种任务
- 可将分类微调模型视为高度专用模型；实践中，打造专用模型往往比打造在多种任务上都表现良好的通用模型更容易

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/03.webp" width=400px>

&nbsp;
### 6.2 准备数据集

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/04.webp" width=500px>

- 本节准备用于分类微调的数据集
- 我们使用由垃圾短信与非垃圾短信组成的文本消息数据集，微调 LLM 对其进行分类
- 首先下载并解压数据集

In [ ]:
import requests
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"


def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} 已存在，跳过下载与解压。")
        return

    # 下载文件
    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()
    with open(zip_path, "wb") as out_file:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                out_file.write(chunk)

    # 解压文件
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    # 添加 .tsv 扩展名
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"文件已下载并保存为 {data_file_path}")


try:
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
except (requests.exceptions.RequestException, TimeoutError) as e:
    print(f"主 URL 失败：{e}，尝试备用 URL…")
    url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/sms%2Bspam%2Bcollection.zip"
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)



# The book originally used the following code below
# However, urllib uses older protocol settings that
# can cause problems for some readers using a VPN.
# The `requests` version above is more robust
# in that regard.

"""
import urllib.request
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} 已存在，跳过下载与解压。")
        return

    # 下载文件
    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    # 解压文件
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    # 添加 .tsv 扩展名
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"文件已下载并保存为 {data_file_path}")

try:
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError) as e:
    print(f"主 URL 失败：{e}，尝试备用 URL…")
    url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/sms%2Bspam%2Bcollection.zip"
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
"""

- 数据集以制表符分隔的文本文件保存，可加载到 pandas DataFrame 中

In [ ]:
import pandas as pd

df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
df

- 查看类别分布可见，数据中「ham」（即「非垃圾短信」）远多于「spam」

In [ ]:
print(df["Label"].value_counts())

- 为简单起见，且我们本就更倾向较小数据集以便教学（可更快完成 LLM 微调），对数据集进行子采样（欠采样），使每个类别各含 747 条样本
- （除欠采样外，处理类别不平衡还有多种方法，但超出 LLM 书籍范围；示例与更多信息见 [`imbalanced-learn` 用户指南](https://imbalanced-learn.org/stable/user_guide.html)）

In [ ]:
def create_balanced_dataset(df):
    
    # 统计 "spam" 实例数
    num_spam = df[df["Label"] == "spam"].shape[0]
    
    # 随机采样与 "spam" 数量相同的 "ham" 实例
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)
    
    # 将 ham 子集与 spam 合并
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])

    return balanced_df


balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

- 接下来，将字符串类别标签「ham」与「spam」映射为整数类别标签 0 与 1：

In [ ]:
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})    

In [ ]:
balanced_df

- 下面定义函数，将数据集随机划分为训练集、验证集与测试集

In [ ]:
def random_split(df, train_frac, validation_frac):
    # 打乱整个 DataFrame
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    # 计算划分索引
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    # 划分 DataFrame
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)
# 测试集占比隐含为剩余 0.2

train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

&nbsp;
### 6.3 创建数据加载器

- 注意文本消息长度不同；若要在 batch 中合并多个训练样本，可以：
  1. 将所有消息截断为数据集或 batch 中最短消息的长度
  2. 将所有消息填充至数据集或 batch 中最长消息的长度

- 我们选择方案 2，将所有消息填充至数据集中最长消息的长度
- 为此使用 `<|endoftext|>` 作为填充 token，如第 2 章所述

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/06.webp" width=500px>

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

- 下方 `SpamDataset` 类会找出训练集中最长序列，并为其他序列添加填充 token 以匹配该长度

In [ ]:
import torch
from torch.utils.data import Dataset


class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        # 预分词文本
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            # 若序列长于 max_length 则截断
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        # 将序列填充至最长序列
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length
        # Note: A more pythonic version to implement this method
        # is the following, which is also used in the next chapter:
        # return max(len(encoded_text) for encoded_text in self.encoded_texts)

In [ ]:
train_dataset = SpamDataset(
    csv_file="train.csv",
    max_length=None,
    tokenizer=tokenizer
)

print(train_dataset.max_length)

- 验证集与测试集同样填充至最长训练序列长度
- 注意：长于最长训练样本的验证/测试样本会在 `SpamDataset` 代码中通过 `encoded_text[:self.max_length]` 被截断
- 该行为完全可选；若在验证集与测试集上将 `max_length=None`，效果同样良好

In [ ]:
val_dataset = SpamDataset(
    csv_file="validation.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)
test_dataset = SpamDataset(
    csv_file="test.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

- 接下来用该数据集实例化 data loader，与前几章创建 data loader 的方式类似

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/07.webp" width=500px>

In [ ]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

- 作为验证步骤，遍历 data loader，确认每个 batch 含 8 个训练样本，且每个样本由 120 个 token 组成

In [ ]:
print("训练加载器:")
for input_batch, target_batch in train_loader:
    pass

print("输入 batch 维度:", input_batch.shape)
print("标签 batch 维度", target_batch.shape)

- 最后，打印各数据集中的 batch 总数

In [ ]:
print(f"{len(train_loader)} 训练 batch")
print(f"{len(val_loader)} 验证 batch")
print(f"{len(test_loader)} 测试 batch")

&nbsp;
### 6.4 用预训练权重初始化模型

- 本节初始化上一章使用的预训练模型

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/08.webp" width=500px>

In [ ]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257,     # 词汇表大小
    "context_length": 1024,  # 上下文长度
    "drop_rate": 0.0,        # Dropout 比率
    "qkv_bias": True         # Query-Key-Value 偏置
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

assert train_dataset.max_length <= BASE_CONFIG["context_length"], (
    f"Dataset length {train_dataset.max_length} exceeds model's context "
    f"length {BASE_CONFIG['context_length']}. Reinitialize data sets with "
    f"`max_length={BASE_CONFIG['context_length']}`"
)

In [ ]:
from gpt_download import download_and_load_gpt2
from previous_chapters import GPTModel, load_weights_into_gpt
# 若本地没有 `previous_chapters.py` 文件，
# 可从 `llms-from-scratch` PyPI 包导入。
# 详情见： https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# 例如：
# from llms_from_scratch.ch04 import GPTModel
# from llms_from_scratch.ch05 import download_and_load_gpt2, load_weights_into_gpt

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval();

- 为确保模型加载正确，再次检查其能否生成连贯文本

In [ ]:
from previous_chapters import (
    generate_text_simple,
    text_to_token_ids,
    token_ids_to_text
)

# 或者：
# from llms_from_scratch.ch05 import (
#    generate_text_simple,
#    text_to_token_ids,
#    token_ids_to_text
# )


text_1 = "Every effort moves you"

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

- 在将模型微调为分类器之前，先看看仅靠 prompting 是否已能分类垃圾短信

In [ ]:
text_2 = (
    "Is the following text 'spam'? Answer with 'yes' or 'no':"
    " 'You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award.'"
)

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_2, tokenizer),
    max_new_tokens=23,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

- 可见模型并不擅长遵循指令
- 这符合预期，因为它仅经预训练、尚未指令微调（指令微调将在下一章介绍）

&nbsp;
### 6.5 添加分类头

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/09.webp" width=500px>

- 本节修改预训练 LLM，使其可用于分类微调
- 先看一下模型架构

In [ ]:
print(model)

- 上方清晰展示了第 4 章实现的架构
- 目标是替换并微调输出层
- 为此先冻结模型，即令所有层不可训练

In [ ]:
for param in model.parameters():
    param.requires_grad = False

- 然后替换输出层（`model.out_head`），其原本将层输入映射到 50,257 维（词表大小）
- 因我们对模型做二分类微调（预测 2 个类别：「spam」与「not spam」），可按下方方式替换输出层；该层默认可训练
- 注意使用 `BASE_CONFIG["emb_dim"]`（在 `"gpt2-small (124M)"` 模型中为 768），使下方代码更通用

In [ ]:
torch.manual_seed(123)

num_classes = 2
model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=num_classes)

- 从技术上讲，仅训练输出层已足够
- 但如我在 [Finetuning Large Language Models](https://magazine.sebastianraschka.com/p/finetuning-large-language-models) 中所述，实验表明微调更多层可明显提升性能
- 因此我们还让最后一个 Transformer 块，以及连接该块与输出层的最终 `LayerNorm` 模块可训练

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/10.webp" width=500px>

In [ ]:
for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True

for param in model.final_norm.parameters():
    param.requires_grad = True

- 仍可与前几章类似地使用该模型
- 例如，向其输入一些文本

In [ ]:
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print("输入:", inputs)
print("输入维度:", inputs.shape) # 形状：(batch_size, num_tokens)

- 与前几章不同的是，它现在有 2 个输出维度，而非 50,257 个

In [ ]:
with torch.no_grad():
    outputs = model(inputs)

print("输出:\n", outputs)
print("输出维度:", outputs.shape) # 形状：(batch_size, num_tokens, num_classes)

- 如前几章所述，每个输入 token 对应一个输出向量
- 因我们输入含 4 个 token 的文本样本，输出由 4 个二维向量组成

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/11.webp" width=500px>

- 第 3 章讨论了注意力机制，它将每个输入 token 与所有其他输入 token 相连
- 第 3 章还介绍了 GPT 类模型使用的因果注意力掩码；该掩码使当前 token 只能关注当前及之前的 token 位置
- 基于该因果注意力机制，第 4（最后）个 token 包含的信息最多，因为它是唯一包含所有其他 token 信息的 token
- 因此我们特别关注该最后 token，并将在垃圾短信分类任务上微调它

In [ ]:
print("最后输出 token:", outputs[:, -1, :])

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/12.webp" width=200px>

&nbsp;
### 6.6 计算分类损失与准确率

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/13.webp" width=300px>

- 在解释损失计算之前，简要说明模型输出如何转为类别标签

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/14.webp" width=600px>

In [ ]:
print("最后输出 token:", outputs[:, -1, :])

- 与第 5 章类似，通过 `softmax` 将输出（logits）转为概率分数，再用 `argmax` 得到最大概率对应的索引

In [ ]:
probas = torch.softmax(outputs[:, -1, :], dim=-1)
label = torch.argmax(probas)
print("类别标签:", label.item())

- 如第 5 章所述，此处 `softmax` 可选，因为最大 logit 即对应最大概率

In [ ]:
logits = outputs[:, -1, :]
label = torch.argmax(logits)
print("类别标签:", label.item())

- 可用该思路计算所谓分类准确率，即数据集中预测正确的比例
- 计算分类准确率时，可对数据集中所有样本应用前述基于 `argmax` 的预测代码，并计算正确预测所占比例，如下：

In [ ]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)

            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]  # 最后输出 token 的 logits
            predicted_labels = torch.argmax(logits, dim=-1)

            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    return correct_predictions / num_examples

- 应用该函数计算各数据集的分类准确率：

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # 使用 PyTorch 2.9 或更高版本以获得稳定的 mps 结果
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print("设备:", device)

model.to(device) # 对 nn.Module 类无需 model = model.to(device) 赋值

torch.manual_seed(123) # 为可复现性（训练 data loader 中有 shuffle）

train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"训练准确率: {train_accuracy*100:.2f}%")
print(f"验证准确率: {val_accuracy*100:.2f}%")
print(f"测试准确率: {test_accuracy*100:.2f}%")

- 可见预测准确率不高，因为我们尚未微调模型

- 在开始微调（/训练）之前，需先定义训练时要优化的损失函数
- 目标是最大化模型的垃圾短信分类准确率；但分类准确率不可微
- 因此改为最小化交叉熵损失，作为最大化分类准确率的代理（更多信息见我的免费课程 [Introduction to Deep Learning](https://sebastianraschka.com/blog/2021/dl-course.html#l08-multinomial-logistic-regression--softmax-regression) 第 8 讲）

- 此处 `calc_loss_batch` 与第 5 章相同，只是我们仅优化最后 token `model(input_batch)[:, -1, :]`，而非所有 token `model(input_batch)`

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, -1, :]  # 最后输出 token 的 logits
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

`calc_loss_loader` 与第 5 章完全相同

In [ ]:
# 与第 5 章相同
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # 若 num_batches 超过 data loader 中的 batch 数，则减少 batch 数
        # 以匹配 data loader 中的 batch 总数
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

- 使用 `calc_loss_loader`，在开始训练前计算训练、验证与测试集的初始损失

In [ ]:
with torch.no_grad(): # 尚未训练，关闭梯度跟踪以提高效率
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)

print(f"训练损失: {train_loss:.3f}")
print(f"验证损失: {val_loss:.3f}")
print(f"测试损失: {test_loss:.3f}")

- 下一节将训练模型以降低损失并提高分类准确率

&nbsp;
### 6.7 在有标注数据上微调模型

- 本节定义并使用训练函数以提高模型分类准确率
- 下方 `train_classifier_simple` 与第 5 章预训练用的 `train_model_simple` 几乎相同
- 仅有两处不同：
  1. 跟踪已见训练样本数（`examples_seen`），而非已见 token 数
  2. 每个 epoch 后计算准确率，而非每个 epoch 后打印样本文本

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/15.webp" width=500px>

In [ ]:
# 总体上与第 5 章的 `train_model_simple` 相同
def train_classifier_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                            eval_freq, eval_iter):
    # 初始化列表以跟踪损失与已见样本数
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1

    # 主训练循环
    for epoch in range(num_epochs):
        model.train()  # 将模型设为训练模式

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # 重置上一 batch 迭代的损失梯度
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # 计算损失梯度
            optimizer.step() # 用损失梯度更新模型权重
            examples_seen += input_batch.shape[0] # 新：跟踪样本数而非 token 数
            global_step += 1

            # 可选评估步骤
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Epoch {epoch+1}（Step {global_step:06d}）："
                      f"训练损失 {train_loss:.3f}，验证损失 {val_loss:.3f}")

        # 每个 epoch 后计算准确率
        train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)
        print(f"训练准确率: {train_accuracy*100:.2f}% | ", end="")
        print(f"验证准确率: {val_accuracy*100:.2f}%")
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

- `train_classifier_simple` 中使用的 `evaluate_model` 与第 5 章相同

In [ ]:
# 与第 5 章相同
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

- 在 M3 MacBook Air 上训练约 5 分钟，在 V100 或 A100 GPU 上不到半分钟

In [ ]:
import time

start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)

num_epochs = 5
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"训练完成，耗时 {execution_time_minutes:.2f} 分钟。")

- 与第 5 章类似，用 matplotlib 绘制训练集与验证集的损失

In [ ]:
import matplotlib.pyplot as plt

def plot_values(epochs_seen, examples_seen, train_values, val_values, label="loss"):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # 绘制训练/验证损失随 epoch 变化
    ax1.plot(epochs_seen, train_values, label=f"训练 {label}")
    ax1.plot(epochs_seen, val_values, linestyle="-.", label=f"验证 {label}")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("损失" if label=="loss" else "准确率")
    ax1.legend()

    # 为已见样本数创建第二个 x 轴
    ax2 = ax1.twiny()  # 创建共享 y 轴的第二个 x 轴
    ax2.plot(examples_seen, train_values, alpha=0)  # 不可见绘图，用于对齐刻度
    ax2.set_xlabel("已见样本数")

    fig.tight_layout()  # 调整布局以留出空间
    plt.savefig(f"{label}-plot.pdf")
    plt.show()

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_losses))

plot_values(epochs_tensor, examples_seen_tensor, train_losses, val_losses)

- 从下降趋势可见模型学习良好
- 训练与验证损失非常接近，说明模型不太过拟合训练数据
- 同样可在下方绘制准确率

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_accs))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_accs))

plot_values(epochs_tensor, examples_seen_tensor, train_accs, val_accs, label="accuracy")

- 从上方准确率图可见，在第 4、5 个 epoch 后训练与验证准确率已相对较高
- 但需记住，我们之前在训练函数中指定了 `eval_iter=5`，即仅估计了训练与验证集表现
- 可在完整数据集上计算训练、验证与测试集表现，如下

In [ ]:
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)

print(f"训练准确率: {train_accuracy*100:.2f}%")
print(f"验证准确率: {val_accuracy*100:.2f}%")
print(f"测试准确率: {test_accuracy*100:.2f}%")

- 可见训练集与验证集表现几乎相同
- 但测试集略低，说明模型对训练数据（以及对用于调整学习率等超参数的验证数据）存在轻微过拟合
- 这很正常；可通过提高模型 dropout 率（`drop_rate`）或优化器中的 `weight_decay` 进一步缩小差距

&nbsp;
### 6.8 将 LLM 用作垃圾短信分类器

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/18.webp" width=500px>

- 最后，使用微调后的 GPT 模型
- 下方 `classify_review` 实现与先前 `SpamDataset` 类似的数据预处理
- 然后返回模型预测的整数类别标签及对应类别名

In [ ]:
def classify_review(text, model, tokenizer, device, max_length=None, pad_token_id=50256):
    model.eval()

    # 准备模型输入
    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[0]
    # 注意：书中此处原误写为 pos_emb.weight.shape[1]
    # 不会破坏代码，但会导致不必要的截断（768 而非 1024）

    # 若序列过长则截断
    input_ids = input_ids[:min(max_length, supported_context_length)]
    assert max_length is not None, (
        "max_length must be specified. If you want to use the full model context, "
        "pass max_length=model.pos_emb.weight.shape[0]."
    )
    assert max_length <= supported_context_length, (
        f"max_length ({max_length}) exceeds model's supported context length ({supported_context_length})."
    )    
    # 或者，下面更健壮的版本能更好地处理 max_length=None 的情况
    # max_len = min(max_length,supported_context_length) if max_length else supported_context_length
    # input_ids = input_ids[:max_len]
    
    # 将序列填充至最长序列
    input_ids += [pad_token_id] * (max_length - len(input_ids))
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0) # 添加 batch 维度

    # 模型推理
    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]  # 最后输出 token 的 logits
    predicted_label = torch.argmax(logits, dim=-1).item()

    # 返回分类结果
    return "spam" if predicted_label == 1 else "not spam"

- 在下方几个示例上试一下

In [ ]:
text_1 = (
    "You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award."
)

print(classify_review(
    text_1, model, tokenizer, device, max_length=train_dataset.max_length
))

In [ ]:
text_2 = (
    "Hey, just wanted to check if we're still on"
    " for dinner tonight? Let me know!"
)

print(classify_review(
    text_2, model, tokenizer, device, max_length=train_dataset.max_length
))

- 最后保存模型，以便日后无需重新训练即可复用

In [ ]:
torch.save(model.state_dict(), "review_classifier.pth")

- 在新会话中，可按如下方式加载模型

In [ ]:
model_state_dict = torch.load("review_classifier.pth", map_location=device, weights_only=True)
model.load_state_dict(model_state_dict)

&nbsp;
## 总结与要点

- 见 [./gpt_class_finetune.py](./gpt_class_finetune.py) 脚本，为分类微调的独立脚本
- 练习解答见 [./exercise-solutions_ch.ipynb](./exercise-solutions_ch.ipynb)
- 此外，感兴趣的读者可在 [附录 E](../../appendix-E) 了解低秩适配（LoRA）等参数高效训练